**Dataset:** Sample - Superstore.csv

**Context:** Retail Analytics

Using **window functions** and **ANSI SQL**, I analyzed a retail dataset to guide business decisions regarding **discount policies, logistics,** and **regional priorities**.
The analysis includes ranking and performance metrics to identify high-impact categories and optimize operational efficiency.

**Skills demonstrated:** SQL window functions, ranking, analytical queries, business optimization
**Language:** SQL

**Questão 1 - Ranking e ordenação por lucro e vendas**


In [19]:
_dntk.execute_sql(
  '-- Questão 1.1 \n-- No código abaixo foi feito um ranking do total de lucros por produto, utilizando a função de janelamento no \n-- campo "Category". As cláusulas "GROUP BY" e "ORDER BY" foram utilizadas para a visualização do dataframe. \n\nSELECT\n    Category, \n    "Product Name", \n    SUM(Profit) AS Total_Profit,\n    RANK() OVER (\n        PARTITION BY Category\n        ORDER BY SUM(Profit) DESC\n    ) AS Profit_Rank\nFROM \'Sample - Superstore.csv\'\nGROUP BY Category, "Product Name"\nORDER BY Category, Profit_Rank;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)

,Category,Product Name,Total_Profit,Profit_Rank
0,Furniture,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",1927.4420,1
1,Furniture,Global Deluxe High-Back Manager's Chair,1558.5910,2
2,Furniture,Hon Pagoda Stacking Chairs,1540.7040,3
3,Furniture,Hon 4070 Series Pagoda Armless Upholstered Sta...,1388.6348,4
4,Furniture,Office Star - Professional Matrix Back Chair w...,1305.6456,5
...,...,...,...,...
1845,Technology,Epson TM-T88V Direct Thermal Printer - Monochr...,-1057.2300,408
1846,Technology,Cisco TelePresence System EX90 Videoconferenci...,-1811.0784,409
1847,Technology,Cubify CubeX 3D Printer Triple Head Print,-3839.9904,410
1848,Technology,Lexmark MX611dhe Monochrome Laser Printer,-4589.9730,411


In [28]:
df_1 = _dntk.execute_sql(
  '-- Questão 1.2\n-- A diferença entre o "RANK()" e o "DENSE_RANK" é que enquanto o primeiro, em uma situação de empate, pula a próxima\n-- posição, o segundo já a contabiliza, ou seja, mantém uma sequência contínua.\nSELECT\n    Category,\n    "Product Name",\n    SUM(Profit) AS Total_Profit,\n    DENSE_RANK() OVER (\n        PARTITION BY Category\n        ORDER BY SUM(Profit) DESC\n    ) AS Profit_Dense_Rank\nFROM \'Sample - Superstore.csv\'\nGROUP BY Category, "Product Name"\nORDER BY Category, Profit_Dense_Rank;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_1

,Category,Product Name,Total_Profit,Profit_Dense_Rank
0,Furniture,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",1927.4420,1
1,Furniture,Global Deluxe High-Back Manager's Chair,1558.5910,2
2,Furniture,Hon Pagoda Stacking Chairs,1540.7040,3
3,Furniture,Hon 4070 Series Pagoda Armless Upholstered Sta...,1388.6348,4
4,Furniture,Office Star - Professional Matrix Back Chair w...,1305.6456,5
...,...,...,...,...
1845,Technology,Epson TM-T88V Direct Thermal Printer - Monochr...,-1057.2300,408
1846,Technology,Cisco TelePresence System EX90 Videoconferenci...,-1811.0784,409
1847,Technology,Cubify CubeX 3D Printer Triple Head Print,-3839.9904,410
1848,Technology,Lexmark MX611dhe Monochrome Laser Printer,-4589.9730,411


In [34]:
df_2 = _dntk.execute_sql(
  '-- Questão 1.3\n-- No código abaixo, utilizamos o "ROW_NUMBER" em janelas definidas pela coluna "Segment" onde a ordenação \n-- é definida pela quantidade de produtos comprados por cada cliente de forma descendente.\n\nSELECT\n    Segment, \n    "Customer Name",\n    SUM(Quantity) AS Total_Quantity,\n    ROW_NUMBER() OVER(\n        PARTITION BY Segment\n        ORDER BY SUM(Quantity) DESC\n    ) AS Quantity_RowNum\nFROM \'Sample - Superstore.csv\'\nGROUP BY Segment, "Customer Name"\nORDER BY Segment, Quantity_RowNum;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_2

,Segment,Customer Name,Total_Quantity,Quantity_RowNum
0,Consumer,William Brown,146.0,1
1,Consumer,John Lee,143.0,2
2,Consumer,Steven Cartwright,133.0,3
3,Consumer,Emily Phan,124.0,4
4,Consumer,Cassandra Brandow,122.0,5
...,...,...,...,...
788,Home Office,Brad Thomas,11.0,144
789,Home Office,Roy Skaria,10.0,145
790,Home Office,Ed Ludwig,9.0,146
791,Home Office,Adrian Shami,9.0,147


**Questão 2 - Análise de variação com LAG e LEAD**

In [40]:
df_3 = _dntk.execute_sql(
  '-- Questão 2.1\n-- No código abaixo, em um primeiro momento o lucro de todos os produtos de um mesmo pedido são somados "SUM(Profit)" \n-- e utilizados no particionamento por "Costumer Name", sendo ordenados por "Order Date" e "Order ID". \n-- Em um segundo momento através da função de janelamento "LAG()", que chama o pedido anterior de um mesmo cliente, é \n-- calculada a diferença entre o pedido atual e o pedido anterior, com o objetivo de verificar a variação de lucro por\n-- cliente.  \n\nSELECT \n    "Customer Name", \n    "Order ID",\n    "Order Date", \n    SUM(Profit) AS Order_Profit, \n    LAG(SUM(Profit)) OVER (\n        PARTITION BY "Customer Name"\n        ORDER BY "Order Date", "Order ID"\n    ) AS Previous_Order_Profit,\n    SUM(Profit) - LAG(SUM(Profit)) OVER (\n        PARTITION BY "Customer Name"\n        ORDER BY "Order Date", "Order ID"\n    ) AS Profit_Diff\nFROM \'Sample - Superstore.csv\'\nGROUP BY "Customer Name", "Order ID", "Order Date"\nORDER BY "Customer Name", "Order Date", "Order ID";',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_3

,Customer Name,Order ID,Order Date,Order_Profit,Previous_Order_Profit,Profit_Diff
0,Aaron Bergman,CA-2014-152905,2014-02-18,-2.5248,NaN,NaN
1,Aaron Bergman,CA-2014-156587,2014-03-07,15.0033,-2.5248,17.5281
2,Aaron Bergman,CA-2016-140935,2016-11-10,116.8680,15.0033,101.8647
3,Aaron Hawkins,CA-2014-122070,2014-04-22,124.7869,NaN,NaN
4,Aaron Hawkins,CA-2014-113768,2014-05-13,24.7992,124.7869,-99.9877
...,...,...,...,...,...,...
5004,Zuschuss Donatelli,CA-2014-143336,2014-08-27,25.8774,NaN,NaN
5005,Zuschuss Donatelli,CA-2016-167682,2016-04-03,146.8280,25.8774,120.9506
5006,Zuschuss Donatelli,US-2016-147991,2016-05-05,3.3440,146.8280,-143.4840
5007,Zuschuss Donatelli,CA-2016-152471,2016-07-08,56.4925,3.3440,53.1485


In [43]:
df_4 = _dntk.execute_sql(
  '-- Questão 2.2\n-- No código abaixo utilizamos a função "LEAD()" para calcular a diferença da quantidade de produtos do próximo \n-- pedido do mesmo cliente e do pedido corrente, analisando assim se o próximo pedido tem mais ou menos produtos do\n-- que o atual. \n\nSELECT\n    "Customer Name", \n    "Order ID", \n    "Order Date",\n    SUM(Quantity) AS Order_Quantity,\n    LEAD(SUM(Quantity)) OVER (\n        PARTITION BY "Customer Name"\n        ORDER BY "Order Date", "Order ID"\n    ) AS Next_Order_Quantity,\n    LEAD(SUM(Quantity)) OVER(\n        PARTITION BY "Customer Name"\n        ORDER BY "Order Date", "Order ID"\n    ) - SUM(Quantity) AS Quantity_Diff\nFROM \'Sample - Superstore.csv\'\nGROUP BY "Customer Name", "Order ID", "Order Date"\nORDER BY "Customer Name", "Order Date", "Order ID";',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_4

,Customer Name,Order ID,Order Date,Order_Quantity,Next_Order_Quantity,Quantity_Diff
0,Aaron Bergman,CA-2014-152905,2014-02-18,2.0,7.0,5.0
1,Aaron Bergman,CA-2014-156587,2014-03-07,7.0,4.0,-3.0
2,Aaron Bergman,CA-2016-140935,2016-11-10,4.0,NaN,NaN
3,Aaron Hawkins,CA-2014-122070,2014-04-22,11.0,8.0,-3.0
4,Aaron Hawkins,CA-2014-113768,2014-05-13,8.0,4.0,-4.0
...,...,...,...,...,...,...
5004,Zuschuss Donatelli,CA-2014-143336,2014-08-27,9.0,8.0,-1.0
5005,Zuschuss Donatelli,CA-2016-167682,2016-04-03,8.0,5.0,-3.0
5006,Zuschuss Donatelli,US-2016-147991,2016-05-05,5.0,7.0,2.0
5007,Zuschuss Donatelli,CA-2016-152471,2016-07-08,7.0,3.0,-4.0


In [49]:
df_5 = _dntk.execute_sql(
  '-- Questão 2.3\n-- No código abaixo utilizamos as funções LAG() e LEAD() na coluna "Sales" para indicar o valor total de vendas do \n-- pedido anterior e do próximo dentro da mesma janela, ou seja, do mesmo cliente. Em seguida utilizamos a expressão\n-- > 500 para identificar os pedidos com variações de vendas superiores a 500. \n\nWITH CustomerSales AS (\n    SELECT\n        "Customer Name",\n        "Order ID",\n        "Order Date",\n        SUM(Sales) AS Order_Sales,\n        LAG(SUM(Sales)) OVER (\n            PARTITION BY "Customer Name"\n            ORDER BY "Order Date", "Order ID"\n        ) AS Previous_Sales,\n        LEAD(SUM(Sales)) OVER (\n            PARTITION BY "Customer Name"\n            ORDER BY "Order Date", "Order ID"\n        ) AS Next_Sales\n    FROM \'Sample - Superstore.csv\'\n    GROUP BY "Customer Name", "Order ID", "Order Date"\n)\nSELECT *\nFROM CustomerSales\nWHERE \n    (Previous_Sales IS NOT NULL AND ABS(Order_Sales - Previous_Sales) > 500)\n    OR\n    (Next_Sales IS NOT NULL AND ABS(Order_Sales - Next_Sales) > 500)\nORDER BY "Customer Name", "Order Date", "Order ID";',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_5

,Customer Name,Order ID,Order Date,Order_Sales,Previous_Sales,Next_Sales
0,Aaron Hawkins,CA-2014-157644,2014-12-31,53.670,49.408,991.260
1,Aaron Hawkins,CA-2015-130113,2015-12-27,991.260,53.670,86.450
2,Aaron Hawkins,CA-2016-162747,2016-03-20,86.450,991.260,18.704
3,Aaron Smayling,CA-2017-113481,2017-01-02,740.214,477.666,1476.270
4,Aaron Smayling,CA-2017-162691,2017-08-01,1476.270,740.214,88.074
...,...,...,...,...,...,...
2268,Zuschuss Carroll,CA-2016-130946,2016-04-08,1616.704,132.709,43.920
2269,Zuschuss Carroll,CA-2017-115322,2017-05-11,43.920,1616.704,4.572
2270,Zuschuss Donatelli,US-2016-147991,2016-05-05,16.720,331.080,839.944
2271,Zuschuss Donatelli,CA-2016-152471,2016-07-08,839.944,16.720,61.440


**Questão 3 - Estatísticas com SUM, COUNT e AVG em janelas**

In [1]:
df_6 = _dntk.execute_sql(
  '-- Questão 3.1\n-- No código abaixo utilizamos o frame "ROWS BETWEEN 2 PRECEDING AND CURRENT ROW" para selecionar o total de lucros\n-- de cada pedido e seus dois pedidos anteriores e assim, calcular a média móvel de lucro particionando por estado.\n-- Como excessão temos o primeiro e o segundo pedido de cada estado/partição. No primeiro somente ele é considerado\n-- e no segundo é considerado ele mesmo e o seu antecessor. \n\nSELECT \n    State,\n    "Order ID", \n    "Order Date",\n    SUM(Profit) AS Order_Profit,\n    AVG(SUM(Profit)) OVER (\n        PARTITION BY State\n        ORDER BY "Order Date", "Order ID"\n        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW\n    ) AS MOVING_Avg_Profit\nFROM \'Sample - Superstore.csv\'\nGROUP BY State, "Order ID", "Order Date"\nORDER BY State, "Order Date", "Order ID";',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_6

,State,Order ID,Order Date,Order_Profit,MOVING_Avg_Profit
0,Alabama,CA-2014-124023,2014-04-07,2.7776,2.777600
1,Alabama,US-2014-118997,2014-04-08,316.1392,159.458400
2,Alabama,CA-2014-143840,2014-05-22,46.5810,121.832600
3,Alabama,CA-2014-110408,2014-10-18,444.6902,269.136800
4,Alabama,CA-2014-163013,2014-11-28,3.9609,165.077367
...,...,...,...,...,...
5004,Wisconsin,CA-2017-102519,2017-11-27,75.3001,41.876700
5005,Wisconsin,CA-2017-117324,2017-12-08,419.3294,178.795567
5006,Wisconsin,CA-2017-155362,2017-12-17,8.4656,167.698367
5007,Wisconsin,CA-2017-143252,2017-12-18,33.8443,153.879767


In [4]:
df_7 = _dntk.execute_sql(
  '-- Questão 3.2\n-- No código abaixo utilizamos o frame "ROWS BETWEEN 4 PRECEDING AND 1 PRECEDING" para somar o total de vendas dos\n-- últimos 4 pedidos anteriores, sem contar com o "current row". Caso não existam 4 exatos pedidos anteriores do \n-- mesmo cliente, são contabilizados aqueles que existirem, ou seja, 1, 2 ou 3. Caso não exista nenhum pedido anterior\n-- o valor associado será "nan".\n\nSELECT\n    "Customer Name",\n    "Order ID",\n    "Order Date",\n    SUM(Sales) AS Order_Sales,\n    SUM(SUM(Sales)) OVER (\n        PARTITION BY "Customer Name"\n        ORDER BY "Order Date", "Order ID"\n        ROWS BETWEEN 4 PRECEDING AND 1 PRECEDING\n    ) AS Sales_Last4Orders\nFROM \'Sample - Superstore.csv\'\nGROUP BY "Customer Name", "Order ID", "Order Date"\nORDER BY "Customer Name", "Order Date", "Order ID";\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_7

,Customer Name,Order ID,Order Date,Order_Sales,Sales_Last4Orders
0,Aaron Bergman,CA-2014-152905,2014-02-18,12.624,NaN
1,Aaron Bergman,CA-2014-156587,2014-03-07,309.592,12.624
2,Aaron Bergman,CA-2016-140935,2016-11-10,563.940,322.216
3,Aaron Hawkins,CA-2014-122070,2014-04-22,257.752,NaN
4,Aaron Hawkins,CA-2014-113768,2014-05-13,287.456,257.752
...,...,...,...,...,...
5004,Zuschuss Donatelli,CA-2014-143336,2014-08-27,244.760,NaN
5005,Zuschuss Donatelli,CA-2016-167682,2016-04-03,331.080,244.760
5006,Zuschuss Donatelli,US-2016-147991,2016-05-05,16.720,575.840
5007,Zuschuss Donatelli,CA-2016-152471,2016-07-08,839.944,592.560


In [4]:
df_8 = _dntk.execute_sql(
  '-- Questão 3.3\n-- No código abaixo foi criada uma CTE com o objetivo de agrupar todos os produtos da mesma subcategoria que aparece\n-- mais de uma vez no mesmo pedido em um único grupo. Em seguida foi aplicada nesta CTE um frame "ROWS BETWEEN\n-- UNBOUNDED PRECEDING AND CURRENT ROW" que contabiliza através do "COUNT("Order ID") quantos pedidos foram feitos\n-- em cada subcategoria.\n\nWITH Orders_By_Subcategory AS (\n    SELECT\n        "Order ID",\n        "Order Date",\n        "Sub-Category"\n    FROM \'Sample - Superstore.csv\'\n    GROUP BY "Order ID", "Order Date", "Sub-Category"\n)\n\nSELECT\n    "Sub-Category",\n    "Order ID", \n    "Order Date",\n    COUNT("Order ID") OVER (\n        PARTITION BY "Sub-Category"\n        ORDER BY "Order Date", "Order ID"\n        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW\n    ) AS Cumulative_Orders\nFROM Orders_By_Subcategory\nORDER BY "Sub-Category", "Order Date", "Order ID";',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_8

,Sub-Category,Order ID,Order Date,Cumulative_Orders
0,Accessories,CA-2014-135405,2014-01-09,1
1,Accessories,CA-2014-162775,2014-01-13,2
2,Accessories,CA-2014-103366,2014-01-15,3
3,Accessories,CA-2014-140795,2014-02-01,4
4,Accessories,CA-2014-107755,2014-02-07,5
...,...,...,...,...
9154,Tables,CA-2017-120376,2017-12-22,303
9155,Tables,CA-2017-142909,2017-12-22,304
9156,Tables,CA-2017-146164,2017-12-22,305
9157,Tables,CA-2017-150910,2017-12-22,306


**Questão 4 - Reutilização de janelas com cláusula WINDOW**

In [25]:
df_9 = _dntk.execute_sql(
  '-- Questão 4.1\n-- No código abaixo foram aplicadas as funções "RANK()" e "ROW_NUMBER" sobre as vendas de cada produto, levando em \n-- conta o particionamento primeiro por "REGION" e depois por "Category".\n\nSELECT\n    Region,\n    Category, \n    "Product Name",\n    Sales,\n    Discount,\n\n    RANK() OVER w_region_produto AS Sales_Rank, \n    ROW_NUMBER() OVER w_region_produto AS Sales_RowNum,\n\n-- Questão 4.2\n-- No código abaixo, foi adicionada a função "AVG()" na janela "w_region_produto" com o objetivo de calcular a média\n-- por produto e região. \n\n    AVG(Discount) OVER w_region_produto AS Avg_Discount\n\nFROM \'Sample - Superstore.csv\'\n\nWINDOW w_region_produto AS (\n    PARTITION BY Region, Category\n    ORDER BY Sales DESC\n)\nORDER BY Region, Category, Sales DESC;\n\n\n-- Questão 4.3\n-- Neste caso, quando escrito em um mesmo bloco no DeepNote, a WINDOW evita repetir a definição de uma janela,\n-- ou seja, podemos definí-la apenas uma vez e reutilizá-la em mais de uma função. Isto acaba deixando o código mais\n-- legível e fácil de entender. \n-- Em relação a performance, esta é melhorada pois o SGBD calcula a janela uma única vez e aplica em múltiplas \n-- funções. Acaba por reduzir o processamento redundante e a repetir "PARTITION BY/ORDER BY para cada função."\n\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_9

,Region,Category,Product Name,Sales,Discount,Sales_Rank,Sales_RowNum,Avg_Discount
0,Central,Furniture,HON 5400 Series Task Chairs for Big and Tall,3504.90,0.0,1,1,0.000000
1,Central,Furniture,Office Star - Professional Matrix Back Chair w...,2807.84,0.0,2,2,0.000000
2,Central,Furniture,Balt Solid Wood Round Tables,2678.94,0.0,3,3,0.000000
3,Central,Furniture,Hon Pagoda Stacking Chairs,2567.84,0.0,4,4,0.000000
4,Central,Furniture,HON 5400 Series Task Chairs for Big and Tall,2453.43,0.3,5,5,0.060000
...,...,...,...,...,...,...,...,...
9989,West,Technology,Cush Cases Heavy Duty Rugged Cover Case for Sa...,7.92,0.2,595,595,0.133893
9990,West,Technology,Maxell 4.7GB DVD-R 5/Pack,7.92,0.0,595,596,0.133893
9991,West,Technology,Kingston Digital DataTraveler 16GB USB 2.0,7.16,0.2,597,597,0.134003
9992,West,Technology,QVS USB Car Charger 2-Port 2.1Amp for iPod/iPh...,5.56,0.2,598,598,0.134114


**Questão 5 - Padrões temporais e segmentações**

In [28]:
df_10 = _dntk.execute_sql(
  '-- Questão 5.1\n-- No código abaixo o DATE_TRUNC() foi aplicado na coluna "Order Date" com o objetivo de chamar apenas o primeiro dia\n-- de cada mês. Através do "RANK()" foi criada uma nova coluna que mostra o rankeamento das vendas por mês de cada \n-- região. \n\nSELECT\n    Region, \n    DATE_TRUNC(\'month\', "Order Date") AS Month,\n    SUM(Sales) AS Monthly_Sales,\n    RANK() OVER (\n        PARTITION BY DATE_TRUNC(\'month\', "Order Date")\n        ORDER BY SUM(Sales) DESC\n    ) AS Sales_Rank\nFROM \'Sample - Superstore.csv\'\nGROUP BY\n    Region,\n    DATE_TRUNC(\'month\', "Order Date")\nORDER BY\n    Month, \n    Sales_Rank;',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_10

,Region,Month,Monthly_Sales,Sales_Rank
0,South,2014-01-01,9322.0920,1
1,West,2014-01-01,2938.7230,2
2,Central,2014-01-01,1539.9060,3
3,East,2014-01-01,436.1740,4
4,South,2014-02-01,2028.9860,1
...,...,...,...,...
187,Central,2017-11-01,15154.9780,4
188,West,2017-12-01,29652.0950,1
189,East,2017-12-01,20084.4160,2
190,Central,2017-12-01,18883.0708,3


In [31]:
df_11 = _dntk.execute_sql(
  '-- Questão 5.2\n-- Através da função "COUNT()" e do frame "ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW" foi possível calcular \n-- quantos pedidos cada cliente fez. \n\nSELECT \n    "Customer Name",\n    "Order ID",\n    "Order Date", \n    COUNT("Order ID") OVER (\n        PARTITION BY "Customer Name"\n        ORDER BY "Order Date", "Order ID"\n        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW\n    ) AS Orders_Cumulative\nFROM \'Sample - Superstore.csv\'\nORDER BY "Customer Name", "Order Date", "Order ID";\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_11

,Customer Name,Order ID,Order Date,Orders_Cumulative
0,Aaron Bergman,CA-2014-152905,2014-02-18,1
1,Aaron Bergman,CA-2014-156587,2014-03-07,2
2,Aaron Bergman,CA-2014-156587,2014-03-07,4
3,Aaron Bergman,CA-2014-156587,2014-03-07,3
4,Aaron Bergman,CA-2016-140935,2016-11-10,5
...,...,...,...,...
9989,Zuschuss Donatelli,CA-2016-167682,2016-04-03,5
9990,Zuschuss Donatelli,US-2016-147991,2016-05-05,6
9991,Zuschuss Donatelli,CA-2016-152471,2016-07-08,7
9992,Zuschuss Donatelli,CA-2016-152471,2016-07-08,8


In [37]:
df_12 = _dntk.execute_sql(
  '-- Questão 5.3\n-- No código abaixo foram criadas duas CTE; na primeira "monthly_profit" é calculada a soma do lucro por mês por \n-- cliente; na segunda "profit_diff" é calculada a diferença de lucro entre o mês corrente e o anterior e uma coluna\n-- "Profit_Growth" é criada para mostrar o crescimento (lucro) que houve por cliente de um mês para o outro. O \n-- filtro "WHERE" foi utilizado de forma a excluir o "NULL" ou seja, o primeiro mês de cada cliente não é contabilizado.\n-- Ao final foram mostrados os cinco clientes que mais tiveram lucro de um mês para o outro.\n\nWITH monthly_profit AS (\n    SELECT\n        "Customer Name",\n        DATE_TRUNC(\'month\', "Order Date") AS Month,\n        SUM(Profit) AS Monthly_Profit\n    FROM \'Sample - Superstore.csv\'\n    GROUP BY "Customer Name", DATE_TRUNC(\'month\', "Order Date")\n),\nprofit_diff AS (\n    SELECT\n        "Customer Name",\n        Month,\n        Monthly_Profit,\n        LAG(Monthly_Profit) OVER (\n            PARTITION BY "Customer Name"\n            ORDER BY Month\n        ) AS Prev_Month_Profit,\n        Monthly_Profit - LAG(Monthly_Profit) OVER (\n            PARTITION BY "Customer Name"\n            ORDER BY Month\n        ) AS Profit_Growth\n    FROM monthly_profit\n)\nSELECT\n    "Customer Name",\n    Month,\n    Monthly_Profit,\n    Prev_Month_Profit,\n    Profit_Growth\nFROM profit_diff\nWHERE Profit_Growth IS NOT NULL\nORDER BY Profit_Growth DESC\nLIMIT 5;\n',
  'SQL_DEEPNOTE_DATAFRAME_SQL',
  audit_sql_comment='',
  sql_cache_mode='cache_disabled',
  return_variable_type='dataframe'
)
df_12

,Customer Name,Month,Monthly_Profit,Prev_Month_Profit,Profit_Growth
0,Tamara Chand,2016-10-01,8762.3891,37.7204,8724.6687
1,Cindy Stewart,2017-11-01,43.7060,-6886.5095,6930.2155
2,Raymond Buch,2017-03-01,6734.4720,72.6159,6661.8561
3,Sanjit Chand,2014-09-01,5511.8641,2.3328,5509.5313
4,Adrian Barton,2016-12-01,4946.3700,-204.4458,5150.8158


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b50e4344-4647-4a93-b699-42e32f41625a' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>